# ⚙️ Notebook 02 — Preprocessing Pipeline

Runs the full preprocessing pipeline and shows results at each step.

**Steps:**
1. Normalize district names
2. Drop kanpur duplicate
3. Convert ERA5 units
4. Winsorize NASA outliers
5. Merge datasets
6. Create targets
7. Feature engineering
8. Train/Val/Test split
9. Scaling

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import warnings
from config_utils import load_config

warnings.filterwarnings('ignore')
cfg = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
print(f'Libraries loaded from {PROJECT_ROOT} ✅')

## Step 1–5: Run Preprocessing Pipeline

In [ ]:
from preprocess import run_preprocessing
merged, train, val, test = run_preprocessing(cfg['_meta']['config_path'])
print(f'\nMerged shape: {merged.shape}')
print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

## Verify Merge Result

In [ ]:
print('Columns in merged dataset:')
print(list(merged.columns))
print(f'\nMissing values after merge:')
print(merged.isnull().sum()[merged.isnull().sum() > 0])
print(f'\nClass balance (rain_today):')
print(merged['rain_today'].value_counts(normalize=True).round(4))

## Step 6: Feature Engineering

In [ ]:
from features import build_features_for_splits, get_feature_columns
train_fe, val_fe, test_fe = build_features_for_splits(train=train, val=val, test=test)
print(f'Shape after feature engineering: {train_fe.shape}')
print(f'New columns added:')
new_cols = [c for c in train_fe.columns if c not in merged.columns]
print(new_cols)

## Step 7: Scaling — RobustScaler

In [ ]:
from features import fit_scaler, apply_scaler, prepare_Xy
feat_cols = get_feature_columns(cfg)
feat_cols = [c for c in feat_cols if c in train_fe.columns]
X_train, y_train = prepare_Xy(train_fe, feat_cols, 'rain_today')
X_val,   y_val   = prepare_Xy(val_fe,   feat_cols, 'rain_today')
scaler = fit_scaler(X_train, cfg)
X_train_sc = apply_scaler(X_train, scaler)
X_val_sc   = apply_scaler(X_val,   scaler)
print(f'X_train scaled shape: {X_train_sc.shape}')
print(f'Feature columns ({len(feat_cols)}):')
print(feat_cols)

## Step 8: SMOTE — Training Set Only

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_res, y_res = smote.fit_resample(X_train_sc, y_train)
print(f'Before SMOTE: {len(X_train_sc):,} rows | Rain: {y_train.mean()*100:.1f}%')
print(f'After SMOTE:  {len(X_res):,} rows | Rain: {y_res.mean()*100:.1f}%')